# Unit 7 practical: a small multimodal app with real models
> **Status: written but NOT executed** while the course was authored (no GPU, no models, no internet-installed libraries on the authoring machine).
> The three models below exist on the Hugging Face hub (checked). Run on Google Colab; if a cell fails, a library version has probably changed.

You will: (1) answer questions about an image with a vision-language model, (2) do **zero-shot image classification** with CLIP and check it against the contrastive idea of the PDF, (3) transcribe speech with Whisper and score it with the **word error rate** from the course code.

In [ ]:
!pip -q install transformers accelerate pillow requests
!wget -q https://krunalpanchal1995.github.io/banu.github.io/downloads/u7/u7-code.zip -O u7.zip && unzip -q -o u7.zip
import sys; sys.path.insert(0, "u7-code/codes/multimodal/multimodal_np"); import multimodal_np as M
import requests, torch
from PIL import Image

## 1. Visual question answering (BLIP)

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering
proc = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base"); vqa = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"   # any image URL or an uploaded file
img = Image.open(requests.get(url, stream=True, headers={"User-Agent": "course-notebook"}).raw).convert("RGB")
for q in ["What animal is this?", "What colour is it?", "Is it indoors?"]:
    out = vqa.generate(**proc(img, q, return_tensors="pt")); print(q, "->", proc.decode(out[0], skip_special_tokens=True))

## 2. Zero-shot classification with CLIP (contrastive learning, Section 3.3.3)
The class names become text prompts; the image is assigned to the prompt with the highest cosine similarity. Compare with `Zi @ prompts.T` in the toy of the PDF.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32"); cp = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]
inp = cp(text=labels, images=img, return_tensors="pt", padding=True)
probs = clip(**inp).logits_per_image.softmax(-1)[0]
print(dict(zip(labels, probs.tolist())))

## 3. Speech to text (Whisper) and word error rate

In [ ]:
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
# upload a short .wav of your own voice reading the reference sentence below
from google.colab import files; up = files.upload(); path = list(up)[0]
hyp = asr(path)["text"].strip().lower()
ref = "the quick brown fox jumps over the lazy dog"
print(hyp); print("WER =", M.wer(ref, hyp))

## Your turn
1. Ask the VQA model a question whose answer is **not** in the image. What does it do (compare Unit 5's abstention)?
2. Add 3 more prompts to the CLIP list, including a misleading one. How confident is the model?
3. Record the sentence in a noisy place and compare WER. Why does WER say nothing about whether the *meaning* survived?
4. Count the image tokens: 224/16 squared. How many does BLIP's vision encoder use? (print the model config).